In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [68]:
import pandas as pd
train = pd.read_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_01/Data/train_cleaned.pkl')
print(train.dtypes)

id                      int64
date           datetime64[ns]
store_nbr               int64
family                 object
sales                 float64
onpromotion             int64
dtype: object


In [69]:
train = train.sort_values(['date', 'store_nbr', 'family'])
train['sales_lag_1'] = train.groupby(['store_nbr', 'family'])['sales'].shift(1)
train['sales_lag_7'] = train.groupby(['store_nbr', 'family'])['sales'].shift(7)

In [70]:
train[(train['store_nbr']==1) & (train['family']=='AUTOMOTIVE')][['date','sales','sales_lag_1','sales_lag_7']].head(10)

,date,sales,sales_lag_1,sales_lag_7
0,2013-01-01,0.0,NaN,NaN
1782,2013-01-02,2.0,0.0,NaN
3564,2013-01-03,3.0,2.0,NaN
5346,2013-01-04,3.0,3.0,NaN
7128,2013-01-05,5.0,3.0,NaN
8910,2013-01-06,2.0,5.0,NaN
10692,2013-01-07,0.0,2.0,NaN
12474,2013-01-08,2.0,0.0,0.0
14256,2013-01-09,2.0,2.0,2.0
16038,2013-01-10,2.0,2.0,3.0


In [71]:
train['rolling_mean_7'] = train.groupby(['family', 'store_nbr'])['sales'].transform(lambda x :x.shift(1).rolling(7).mean())
train[(train['store_nbr'] == 1) & (train['family']== 'AUTOMOTIVE')][['date', 'family', 'sales', 'sales_lag_1','sales_lag_7', 'rolling_mean_7']].head(10)

,date,family,sales,sales_lag_1,sales_lag_7,rolling_mean_7
0,2013-01-01,AUTOMOTIVE,0.0,NaN,NaN,NaN
1782,2013-01-02,AUTOMOTIVE,2.0,0.0,NaN,NaN
3564,2013-01-03,AUTOMOTIVE,3.0,2.0,NaN,NaN
5346,2013-01-04,AUTOMOTIVE,3.0,3.0,NaN,NaN
7128,2013-01-05,AUTOMOTIVE,5.0,3.0,NaN,NaN
8910,2013-01-06,AUTOMOTIVE,2.0,5.0,NaN,NaN
10692,2013-01-07,AUTOMOTIVE,0.0,2.0,NaN,NaN
12474,2013-01-08,AUTOMOTIVE,2.0,0.0,0.0,2.142857
14256,2013-01-09,AUTOMOTIVE,2.0,2.0,2.0,2.428571
16038,2013-01-10,AUTOMOTIVE,2.0,2.0,3.0,2.428571


In [72]:
train['Month'] = train['date'].dt.month
train['Year'] = train['date'].dt.year
train['is_weekend'] = train['date'].dt.dayofweek.isin([5,6]).astype(int)
train.head(10)

,id,date,store_nbr,family,sales,onpromotion,sales_lag_1,sales_lag_7,rolling_mean_7,Month,Year,is_weekend
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,NaN,NaN,NaN,1,2013,0
1,1,2013-01-01,1,BABY CARE,0.0,0,NaN,NaN,NaN,1,2013,0
2,2,2013-01-01,1,BEAUTY,0.0,0,NaN,NaN,NaN,1,2013,0
3,3,2013-01-01,1,BEVERAGES,0.0,0,NaN,NaN,NaN,1,2013,0
4,4,2013-01-01,1,BOOKS,0.0,0,NaN,NaN,NaN,1,2013,0
5,5,2013-01-01,1,BREAD/BAKERY,0.0,0,NaN,NaN,NaN,1,2013,0
6,6,2013-01-01,1,CELEBRATION,0.0,0,NaN,NaN,NaN,1,2013,0
7,7,2013-01-01,1,CLEANING,0.0,0,NaN,NaN,NaN,1,2013,0
8,8,2013-01-01,1,DAIRY,0.0,0,NaN,NaN,NaN,1,2013,0
9,9,2013-01-01,1,DELI,0.0,0,NaN,NaN,NaN,1,2013,0


In [73]:
launch_dates = train[train['sales']>0].groupby(['family', 'store_nbr'])['date'].min().rename('launch_date')
train = train.merge(launch_dates, on = ['family', 'store_nbr'], how= 'left')
train['is_launched'] = (train['date'] >= train['launch_date']).astype(int)
train['days_since_launched'] = (train['date'] - train['launch_date']).dt.days
train['days_since_launched'] = train['days_since_launched'].clip(lower=0)
train.head()

,id,date,store_nbr,family,sales,onpromotion,sales_lag_1,sales_lag_7,rolling_mean_7,Month,Year,is_weekend,launch_date,is_launched,days_since_launched
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,NaN,NaN,NaN,1,2013,0,2013-01-02,0,0.0
1,1,2013-01-01,1,BABY CARE,0.0,0,NaN,NaN,NaN,1,2013,0,NaT,0,NaN
2,2,2013-01-01,1,BEAUTY,0.0,0,NaN,NaN,NaN,1,2013,0,2013-01-02,0,0.0
3,3,2013-01-01,1,BEVERAGES,0.0,0,NaN,NaN,NaN,1,2013,0,2013-01-02,0,0.0
4,4,2013-01-01,1,BOOKS,0.0,0,NaN,NaN,NaN,1,2013,0,2016-10-13,0,0.0


In [78]:
taking_idea = train[(train['store_nbr'] ==1) & (train['family'] == 'AUTOMOTIVE')][['date', 'sales', 'sales_lag_1', 'launch_date', 'is_launched', 'days_since_launched']]
taking_idea.head(10)

,date,sales,sales_lag_1,launch_date,is_launched,days_since_launched
0,2013-01-01,0.0,NaN,2013-01-02,0,0.0
1782,2013-01-02,2.0,0.0,2013-01-02,1,0.0
3564,2013-01-03,3.0,2.0,2013-01-02,1,1.0
5346,2013-01-04,3.0,3.0,2013-01-02,1,2.0
7128,2013-01-05,5.0,3.0,2013-01-02,1,3.0
8910,2013-01-06,2.0,5.0,2013-01-02,1,4.0
10692,2013-01-07,0.0,2.0,2013-01-02,1,5.0
12474,2013-01-08,2.0,0.0,2013-01-02,1,6.0
14256,2013-01-09,2.0,2.0,2013-01-02,1,7.0
16038,2013-01-10,2.0,2.0,2013-01-02,1,8.0


In [79]:
train.to_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_01/Data/train_features.pkl')